# Phase 4 — Author-aligned Figure 3 experiment (C1 and C2)

**Read first:** This is a separate experimental branch, not a replacement for Phase 3. The first author's public `network_zoo.py` has stride-1 C1 convolutions (contrary to Appendix B.3's stride-2 notation), and convolutional dropout in C2. The notebook retrains **C1/C2 SB and LB** from scratch with those two architectural changes. F1 is **not retrained** because the inspected author repository does not provide an F1 reference architecture.

The notebook reads the repository's **single root `requirements.txt`**. It saves checkpoints every five epochs on Drive; restarting Colab can resume by copying those checkpoints back. Start with the **benchmark cells and share their outputs** before launching expensive training. All plots derive from measured JSONs, not fixed data points. Phase 3 results are never overwritten.

Author reference: https://github.com/keskarnitish/large-batch-training/blob/master/network_zoo.py

## 1. Clone or pull your private repo (GitHub token in Colab Secrets)

In [ ]:
from google.colab import userdata
from pathlib import Path
import subprocess, os, base64
repo = 'vepaan/advanced-ml-paper-reproductions'
repo_dir = Path('/content/advanced-ml-paper-reproductions')
token = userdata.get('GITHUB_TOKEN')
assert token, 'Set GITHUB_TOKEN in Colab Secrets and allow notebook access'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy()
env['GIT_CONFIG_COUNT'] = '1'
env['GIT_CONFIG_KEY_0'] = 'http.extraheader'
env['GIT_CONFIG_VALUE_0'] = f'AUTHORIZATION: basic {auth}'
try:
    if repo_dir.exists():
        subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'], check=True, env=env)
    else:
        subprocess.run(['git','clone',f'https://github.com/{repo}.git',str(repo_dir)], check=True, env=env)
finally:
    del token, auth
print('Commit:')
subprocess.run(['git','-C',str(repo_dir),'log','-1','--oneline'],check=True)

## 2. Mount Drive and use the project-wide dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, subprocess
from pathlib import Path
REPO = Path('/content/advanced-ml-paper-reproductions')
PROJECT = REPO / 'paper2-large-batch'
assert (REPO / 'requirements.txt').is_file()
assert (PROJECT / 'src/phase4_models.py').is_file(), 'First copy the Phase 4 source files into your private Git repo and push.'
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'requirements.txt')], check=True)
import torch
assert torch.cuda.is_available(), 'Select an A100 GPU runtime under Runtime → Change runtime type.'
print('CUDA:',torch.cuda.get_device_name(0),'torch:',torch.__version__)
DRIVE_ROOT = Path('/content/drive/MyDrive/ML-Paper-Reproductions/paper2/phase4_author_aligned')
LOCAL_ROOT = Path('/content/paper2_phase4_author_aligned')
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
LOCAL_ROOT.mkdir(parents=True,exist_ok=True)
os.chdir(PROJECT)
print('Working directory:',Path.cwd())

## 3. Validate CIFAR-10 and inspect the new architecture

In [ ]:
import sys, subprocess
from pathlib import Path
cifar = REPO / 'paper1-generalization/data/cifar-10-batches-py'
if not (cifar / 'data_batch_1').is_file():
    from torchvision.datasets import CIFAR10
    print('Downloading CIFAR-10 into the Paper 1 shared data directory...')
    CIFAR10(root=str(cifar.parent),train=True,download=True)
    CIFAR10(root=str(cifar.parent),train=False,download=True)
assert (cifar / 'data_batch_1').is_file()
for file in ['phase4_models.py','phase4_train.py','phase4_interpolate.py']:
    subprocess.run([sys.executable,'-m','py_compile','src/'+file],check=True)
subprocess.run([sys.executable,'-u','src/phase4_models.py'],check=True)
subprocess.run([sys.executable,'-u','src/data.py','--panel','C1'],check=True)
print('Validation complete. Send this cell output if any parameter counts/shapes are surprising.')

## 4. Benchmark all four training configurations — **STOP AND SEND THIS OUTPUT**

In [ ]:
import subprocess,sys
for panel in ('C1','C2'):
    for regime in ('SB','LB'):
        print(f'\n===== BENCHMARK {panel} {regime} =====',flush=True)
        subprocess.run([sys.executable,'-u','src/phase4_train.py',
                        '--panel',panel,'--regime',regime,'--mode','benchmark',
                        '--device','cuda','--bench-steps','8'],check=True)
print('BENCHMARKS COMPLETE. Share these results before launching training.')

## 5. Training cells — run **one at a time** after benchmarks

Each cell trains to **50 total epochs** (not 50 more after resume) using Adam 0.001 and the exact paper batch sizes (256 SB / 5000 LB). We use a fixed initial budget rather than stopping early based on a 2,048-image probe. If the training set is not adequately fit, **do not blindly extend**: inspect metrics and decide a recorded Phase 4 continuation (e.g. 0.0002 LR) first. Each checkpoint (`last.pt`) is backed up every 5 epochs. The final `endpoint.pt` and `metrics.json` also go to Drive.

If Colab restarts, re-run setup/validation and the restore cell below. It copies only Phase 4 checkpoints, never Phase 3.

In [ ]:
import shutil
for panel in ('C1','C2'):
    for regime in ('SB','LB'):
        name=f'{panel}_{regime}_seed42'
        local=LOCAL_ROOT/name
        remote=DRIVE_ROOT/name
        if remote.is_dir() and (remote/'last.pt').is_file() and not (local/'last.pt').is_file():
            shutil.copytree(remote, local, dirs_exist_ok=True)
            print('Restored',name)
print('Restore step complete.')

### Train C1 SB — separate resumable experiment

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-u','src/phase4_train.py',
                '--mode','train','--panel','C1','--regime','SB',
                '--device','cuda','--seed','42',
                '--max-epochs','50','--checkpoint-every','5',
                '--output-root',str(LOCAL_ROOT),'--backup-root',str(DRIVE_ROOT)],check=True)
print('Finished C1 SB. Check the reported full-dataset train accuracy.')

### Train C1 LB — separate resumable experiment

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-u','src/phase4_train.py',
                '--mode','train','--panel','C1','--regime','LB',
                '--device','cuda','--seed','42',
                '--max-epochs','50','--checkpoint-every','5',
                '--output-root',str(LOCAL_ROOT),'--backup-root',str(DRIVE_ROOT)],check=True)
print('Finished C1 LB. Check the reported full-dataset train accuracy.')

### Train C2 SB — separate resumable experiment

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-u','src/phase4_train.py',
                '--mode','train','--panel','C2','--regime','SB',
                '--device','cuda','--seed','42',
                '--max-epochs','50','--checkpoint-every','5',
                '--output-root',str(LOCAL_ROOT),'--backup-root',str(DRIVE_ROOT)],check=True)
print('Finished C2 SB. Check the reported full-dataset train accuracy.')

### Train C2 LB — separate resumable experiment

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-u','src/phase4_train.py',
                '--mode','train','--panel','C2','--regime','LB',
                '--device','cuda','--seed','42',
                '--max-epochs','50','--checkpoint-every','5',
                '--output-root',str(LOCAL_ROOT),'--backup-root',str(DRIVE_ROOT)],check=True)
print('Finished C2 LB. Check the reported full-dataset train accuracy.')

## 6. Inspect all endpoint metrics BEFORE interpolation

In [ ]:
import json
from pathlib import Path
for panel in ('C1','C2'):
    for regime in ('SB','LB'):
        name=f'{panel}_{regime}_seed42'
        f=LOCAL_ROOT/name/'metrics.json'
        assert f.is_file(),f'Missing {f}. Complete that run first.'
        x=json.loads(f.read_text())
        print(f"{name}: epochs={x['epoch']}  trainCE={x['train']['loss']:.5f} "
              f"trainAcc={100*x['train']['accuracy']:.3f}%  "
              f"testAcc={100*x['test']['accuracy']:.3f}%")
print('Send this output before deciding whether to continue training or interpolate.')

## 7. Interpolate each new pair (61 points, BN recalibration)

Run these only when we accept the trained endpoints. The Phase 4 wrapper calls the same interpolation method used in Phase 3, but constructs the author-aligned models. It evaluates all 50k train and 10k test images at each α, with BatchNorm statistics recalibrated on the first 4,096 train images. Each result is saved to Drive immediately.

### C1 61-point interpolation

In [ ]:
import subprocess,sys,shutil,json
panel='C1'
for regime in ('SB','LB'):
    assert (LOCAL_ROOT/f'{panel}_{regime}_seed42'/'endpoint.pt').is_file()
subprocess.run([sys.executable,'-u','src/phase4_interpolate.py',
                '--panel',panel,'--device','cuda','--output-root',str(LOCAL_ROOT),
                '--points','61','--bn-policy','recalibrate','--bn-examples','4096',
                '--eval-batch','512'],check=True)
source=LOCAL_ROOT/f'{panel}_interpolation.json'
result=json.loads(source.read_text())
assert len(result['points'])==61
shutil.copy2(source,DRIVE_ROOT/source.name)
print('Saved Phase 4',panel,'interpolation to Drive')

### C2 61-point interpolation

In [ ]:
import subprocess,sys,shutil,json
panel='C2'
for regime in ('SB','LB'):
    assert (LOCAL_ROOT/f'{panel}_{regime}_seed42'/'endpoint.pt').is_file()
subprocess.run([sys.executable,'-u','src/phase4_interpolate.py',
                '--panel',panel,'--device','cuda','--output-root',str(LOCAL_ROOT),
                '--points','61','--bn-policy','recalibrate','--bn-examples','4096',
                '--eval-batch','512'],check=True)
source=LOCAL_ROOT/f'{panel}_interpolation.json'
result=json.loads(source.read_text())
assert len(result['points'])==61
shutil.copy2(source,DRIVE_ROOT/source.name)
print('Saved Phase 4',panel,'interpolation to Drive')

## 8. Compare final Phase 3 and Phase 4 curves (inline and PNG/PDF)

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt, shutil
from pathlib import Path
PHASE3 = Path('/content/drive/MyDrive/ML-Paper-Reproductions/paper2/final_61')
figure_dir=LOCAL_ROOT/'figures'
figure_dir.mkdir(exist_ok=True)
label=dict(F1='a',C1='c',C2='d')
fig,axes=plt.subplots(1,3,figsize=(15,4.5))
for ax,panel in zip(axes,['F1','C1','C2']):
    source=(PHASE3 if panel=='F1' else LOCAL_ROOT)/f'{panel}_interpolation.json'
    assert source.is_file(),f'Not found: {source}'
    d=json.loads(source.read_text()); pts=sorted(d['points'],key=lambda x:x['alpha'])
    a=np.array([x['alpha'] for x in pts]);
    assert len(pts)==61 and np.any(np.isclose(a,0)) and np.any(np.isclose(a,1))
    losses=[np.array([x[k] for x in pts]) for k in ('train_loss','test_loss')]
    accs=[100*np.array([x[k] for x in pts]) for k in ('train_accuracy','test_accuracy')]
    ax.plot(a,losses[0],'b-',lw=1.8,label='Train')
    ax.plot(a,losses[1],'b--',lw=1.8,label='Test')
    ax.set(xlim=(-1,2),xlabel='alpha',ylabel='Cross Entropy')
    ax.tick_params(axis='y',labelcolor='blue')
    ax.grid(alpha=.25,ls=':')
    other=ax.twinx()
    other.plot(a,accs[0],'r-',lw=1.8)
    other.plot(a,accs[1],'r--',lw=1.8)
    other.set(ylim=(0,100),ylabel='Accuracy (%)')
    other.tick_params(axis='y',labelcolor='red')
    ax.set_title(f'({label[panel]}) {panel}' + (' [Phase 3]' if panel=='F1' else ' [Phase 4]'),y=-.32)
    ax.legend(loc='upper left',fontsize=9)
fig.subplots_adjust(left=.06,right=.94,bottom=.23,top=.95,wspace=.50)
for ext in ('png','pdf'):
    dst=figure_dir/f'figure3_phase4_author_aligned.{ext}'
    fig.savefig(dst,dpi=300,bbox_inches='tight')
    shutil.copy2(dst,DRIVE_ROOT/dst.name)
plt.show();plt.close(fig)
print('Saved final figure:',figure_dir)

## 9. Quantitative endpoint comparison: original versus Phase 4

In [ ]:
for panel in ('C1','C2'):
    old=json.loads((PHASE3/f'{panel}_interpolation.json').read_text())
    new=json.loads((LOCAL_ROOT/f'{panel}_interpolation.json').read_text())
    print(f'\n{panel}:')
    for regime,target in [('SB',0.),('LB',1.)]:
        x=next(p for p in old['points'] if abs(p['alpha']-target)<1e-8)
        y=next(p for p in new['points'] if abs(p['alpha']-target)<1e-8)
        print(f'  {regime}: Phase3 train/test = {x["train_accuracy"]:.3%}/{x["test_accuracy"]:.3%}; '
              f'Phase4 = {y["train_accuracy"]:.3%}/{y["test_accuracy"]:.3%}; '
              f'Phase4 test CE={y["test_loss"]:.5f}')